# F98 – SQUID: Data analysis

All paths are relative to the repository root. Measurement data are read from `F98/Emilian_Fritz`, figures are saved to `plots/`.

In [ ]:
# Imports
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from IPython.display import display, Math

# Relative paths
DATA_DIR = Path("F98") / "Emilian_Fritz"  # measurement data
PLOT_DIR = Path("plots")  # saved figures
PLOT_DIR.mkdir(exist_ok=True)

# Constants
K_BOLTZMANN = 1.380649e-23  # J/K
V_GAIN_521 = 2000  # voltage amplifier gain in 5.2.1

## Uncertainties

The fit uncertainties alone are very small because every fit uses thousands of points. The following uncertainties are therefore taken into account:

* **Input uncertainties of the data points:** the oscilloscope digitises with a finite resolution $q$ (smallest step between two values). Assuming a uniform distribution, every point has $\sigma = q/\sqrt{12}$. Uncertainties in $x$ are included with the effective-variance method $\sigma_\mathrm{eff}^2 = \sigma_y^2 + (m\,\sigma_x)^2$. If $\chi^2_\mathrm{red} > 1$, the scatter is larger than the input uncertainties explain, and the parameter uncertainties are scaled by $\sqrt{\chi^2_\mathrm{red}}$.
* **Current calibration:** $U_{pp}$ of the ramp is read from the maximum and minimum of the digitised signal, both uncertain by one resolution step. The relative uncertainty $\Delta U_{pp}/U_{pp}$ propagates into every calibrated current.
* **Repeat measurements:** where two measurements (`_1`, `_2`) exist, both are evaluated. The mean is used and half the difference is added as reproducibility uncertainty.
* **Fit range (5.3.2):** the fit is repeated with different fit ranges, and the largest deviation is added as systematic uncertainty.

All components are added in quadrature.

In [ ]:
# Helper functions for the uncertainty analysis

def resolution(x):
    """Smallest step between two distinct values (digitisation resolution)."""
    steps = np.diff(np.unique(x))
    return steps[steps > 0].min()


def calibrate_current(U_ramp, I_pp):
    """Calibration factor (µA/V) of a current ramp and its relative uncertainty.

    U_pp is read from the maximum and minimum of the ramp, which are both uncertain
    by one resolution step of the oscilloscope.
    """
    U_pp = U_ramp.max() - U_ramp.min()  # V
    dU_pp = resolution(U_ramp)  # V
    return I_pp / U_pp, dU_pp / U_pp


def weighted_linear_fit(x, y, sigma_x, sigma_y):
    """Linear fit y = m x + b with input uncertainties (effective-variance method).

    Returns m, b, dm, db and chi²_red. For chi²_red > 1 the parameter uncertainties
    are scaled by sqrt(chi²_red).
    """
    m_0, b_0 = np.polyfit(x, y, 1)
    sigma = np.full_like(y, np.sqrt(sigma_y**2 + (m_0 * sigma_x)**2), dtype=float)
    popt, pcov = curve_fit(lambda x, m, b: m * x + b, x, y, p0=[m_0, b_0], sigma=sigma, absolute_sigma=True)
    chi2_red = np.sum(((y - (popt[0] * x + popt[1])) / sigma)**2) / (len(x) - 2)
    dm, db = np.sqrt(np.diag(pcov)) * np.sqrt(max(1.0, chi2_red))
    return popt[0], popt[1], dm, db, chi2_red


def combine_repeats(values, d_stat, rel_cal):
    """Mean of repeat measurements with statistical, calibration and reproducibility uncertainty."""
    values = np.asarray(values)
    mean = values.mean()
    u_stat = np.sqrt(np.sum(np.asarray(d_stat)**2)) / len(values)
    u_cal = np.abs(mean) * np.mean(rel_cal)
    u_rep = (values.max() - values.min()) / 2
    u_tot = np.sqrt(u_stat**2 + u_cal**2 + u_rep**2)
    return mean, u_tot, u_stat, u_cal, u_rep


def fit_IV_resistance(path, I_pp, t_min, t_max, I1, I2):
    """Normal-state resistance from the linear part of an I-V characteristic."""
    t, U, V = np.loadtxt(path, skiprows=3).T  # ms, V, mV
    cal, rel_cal = calibrate_current(U, I_pp)
    I = U * cal  # µA
    mask = (t >= t_min) & (t <= t_max) & (I >= I1) & (I <= I2)
    sigma_I = resolution(U) / np.sqrt(12) * cal  # µA
    sigma_V = resolution(V) / np.sqrt(12)  # mV
    m, b, dm, db, chi2_red = weighted_linear_fit(I[mask], V[mask], sigma_I, sigma_V)
    R = m * 1000 / V_GAIN_521  # Ω
    dR = dm * 1000 / V_GAIN_521  # Ω
    return dict(m=m, b=b, dm=dm, db=db, chi2_red=chi2_red, R=R, dR=dR, rel_cal=rel_cal)

# 5.2.1 – Characteristics of the single SQUID

## I-V characteristic at integer flux

In [ ]:
# Load measurement data (time in ms, channel A: current ramp in V, channel B: voltage in mV)
data_int = np.loadtxt(
    DATA_DIR / "5_2_1" / "30A_voltage_char_int" / "30A_voltage_char_int_2.txt",
    skiprows=3,
)

# Time interval (ms)
t_min_int = -10
t_max_int = 10

# Calibrate current: the peak-to-peak ramp corresponds to I_pp = 30 µA
I_pp_int = 30  # µA
U_pp_int = data_int[:, 1].max() - data_int[:, 1].min()  # V
cal_int = I_pp_int / U_pp_int  # µA/V

# Extract measurement data
t_int = data_int[:, 0]  # ms
I_int = data_int[:, 1] * cal_int  # µA
V_int = data_int[:, 2]  # mV

# Select data within the time interval
mask_int = (t_int >= t_min_int) & (t_int <= t_max_int)
I_sel_int = I_int[mask_int]
V_sel_int = V_int[mask_int]

# Fit boundaries (µA)
I1_int = 5.5
I2_int = 14

# Plot: I-V characteristic
plt.figure(figsize=(8, 5))
plt.plot(I_sel_int, V_sel_int, marker=".", markersize=0.1, color="blue")
plt.axvline(I1_int, linestyle="--", linewidth=0.8, color="red", label="Fit boundaries")
plt.axvline(I2_int, linestyle="--", linewidth=0.8, color="red")
plt.xlabel("Current I (µA)")
plt.ylabel("Voltage V (mV)")
plt.title("I-V characteristic (integer flux)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PLOT_DIR / "plot1.png", format="png")
plt.show()

In [ ]:
# Fit both repeat measurements with input uncertainties
fits_int = [
    fit_IV_resistance(
        DATA_DIR / "5_2_1" / "30A_voltage_char_int" / f"30A_voltage_char_int_{n}.txt",
        I_pp_int, t_min_int, t_max_int, I1_int, I2_int,
    )
    for n in (1, 2)
]
for n, fit in zip((1, 2), fits_int):
    print(f"Measurement {n}: m = ({fit['m']:.4f} ± {fit['dm']:.4f}) mV/µA, "
          f"R_N = ({fit['R']:.4f} ± {fit['dR']:.4f}) Ω, chi²_red = {fit['chi2_red']:.2f}")

# Plot: linear fit of the plotted measurement (_2)
fit_2 = fits_int[1]
mask_fit_int = mask_int & (I_int >= I1_int) & (I_int <= I2_int)
I_fit_int = I_int[mask_fit_int]
V_fit_int = V_int[mask_fit_int]
I_line_int = np.linspace(I_fit_int.min(), I_fit_int.max(), 100)

plt.figure(figsize=(8, 5))
plt.scatter(I_fit_int, V_fit_int, s=10, marker=".", color="blue", label="Measured data")
plt.plot(I_line_int, fit_2["m"] * I_line_int + fit_2["b"], color="red", label="Linear fit")
plt.xlabel("Current I (µA)")
plt.ylabel("Voltage V (mV)")
plt.title("Linear fit of the resistive branch (integer flux)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PLOT_DIR / "plot2.png", format="png")
plt.show()

# Normal-state resistance: mean of both measurements
R_int, dR_int, u_stat, u_cal, u_rep = combine_repeats(
    [fit["R"] for fit in fits_int], [fit["dR"] for fit in fits_int], [fit["rel_cal"] for fit in fits_int]
)
print(f"Normal-state resistance: R_N = ({R_int:.3f} ± {dR_int:.3f}) Ω")
print(f"  (stat. {u_stat:.4f} Ω, calibration {u_cal:.4f} Ω, reproducibility {u_rep:.4f} Ω)")

## I-V characteristic at half-integer flux

In [ ]:
# Load measurement data (time in ms, channel A: current ramp in V, channel B: voltage in mV)
data_half = np.loadtxt(
    DATA_DIR / "5_2_1" / "30A_voltage_char_half" / "30A_voltage_char_half_2.txt",
    skiprows=3,
)

# Time interval (ms)
t_min_half = -10
t_max_half = 10

# Calibrate current: the peak-to-peak ramp corresponds to I_pp = 30 µA
I_pp_half = 30  # µA
U_pp_half = data_half[:, 1].max() - data_half[:, 1].min()  # V
cal_half = I_pp_half / U_pp_half  # µA/V

# Extract measurement data
t_half = data_half[:, 0]  # ms
I_half = data_half[:, 1] * cal_half  # µA
V_half = data_half[:, 2]  # mV

# Select data within the time interval
mask_half = (t_half >= t_min_half) & (t_half <= t_max_half)
I_sel_half = I_half[mask_half]
V_sel_half = V_half[mask_half]

# Fit boundaries (µA)
I1_half = -0.5
I2_half = 10

# Plot: I-V characteristic
plt.figure(figsize=(8, 5))
plt.plot(I_sel_half, V_sel_half, marker=".", markersize=0.1, color="blue")
plt.axvline(I1_half, linestyle="--", linewidth=0.8, color="red", label="Fit boundaries")
plt.axvline(I2_half, linestyle="--", linewidth=0.8, color="red")
plt.xlabel("Current I (µA)")
plt.ylabel("Voltage V (mV)")
plt.title("I-V characteristic (half-integer flux)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PLOT_DIR / "plot3.png", format="png")
plt.show()

In [ ]:
# Fit both repeat measurements with input uncertainties
fits_half = [
    fit_IV_resistance(
        DATA_DIR / "5_2_1" / "30A_voltage_char_half" / f"30A_voltage_char_half_{n}.txt",
        I_pp_half, t_min_half, t_max_half, I1_half, I2_half,
    )
    for n in (1, 2)
]
for n, fit in zip((1, 2), fits_half):
    print(f"Measurement {n}: m = ({fit['m']:.4f} ± {fit['dm']:.4f}) mV/µA, "
          f"R_N = ({fit['R']:.4f} ± {fit['dR']:.4f}) Ω, chi²_red = {fit['chi2_red']:.2f}")

# Plot: linear fit of the plotted measurement (_2)
fit_2 = fits_half[1]
mask_fit_half = mask_half & (I_half >= I1_half) & (I_half <= I2_half)
I_fit_half = I_half[mask_fit_half]
V_fit_half = V_half[mask_fit_half]
I_line_half = np.linspace(I_fit_half.min(), I_fit_half.max(), 100)

plt.figure(figsize=(8, 5))
plt.scatter(I_fit_half, V_fit_half, s=10, marker=".", color="blue", label="Measured data")
plt.plot(I_line_half, fit_2["m"] * I_line_half + fit_2["b"], color="red", label="Linear fit")
plt.xlabel("Current I (µA)")
plt.ylabel("Voltage V (mV)")
plt.title("Linear fit of the resistive branch (half-integer flux)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PLOT_DIR / "plot4.png", format="png")
plt.show()

# Normal-state resistance: mean of both measurements
R_half, dR_half, u_stat, u_cal, u_rep = combine_repeats(
    [fit["R"] for fit in fits_half], [fit["dR"] for fit in fits_half], [fit["rel_cal"] for fit in fits_half]
)
print(f"Normal-state resistance: R_N = ({R_half:.3f} ± {dR_half:.3f}) Ω")
print(f"  (stat. {u_stat:.4f} Ω, calibration {u_cal:.4f} Ω, reproducibility {u_rep:.4f} Ω)")

## Resistance of the SQUID $R_{\mathrm{SQUID}}$ and of a single Josephson junction $R_{\mathrm{J}}$

In [ ]:
# Mean of integer and half-integer flux
R_SQUID = (R_int + R_half) / 2  # Ω
dR_SQUID = np.sqrt(dR_int**2 + dR_half**2) / 2  # Ω

# Two junctions in parallel: R_J = 2 R_SQUID
R_J = 2 * R_SQUID  # Ω
dR_J = 2 * dR_SQUID  # Ω

print(f"SQUID resistance:              R_SQUID = ({R_SQUID:.3f} ± {dR_SQUID:.3f}) Ω")
print(f"Josephson junction resistance: R_J     = ({R_J:.3f} ± {dR_J:.3f}) Ω")

## Critical current $I_c$

In [ ]:
# Wider time interval of the integer-flux measurement (ms)
t_min_Ic = -30
t_max_Ic = 10

# Select data within the time interval
mask_Ic = (t_int >= t_min_Ic) & (t_int <= t_max_Ic)
t_sel_Ic = t_int[mask_Ic]
V_sel_Ic = V_int[mask_Ic]

# Times of the transitions into the voltage state (ms)
t1_Ic = -17
t2_Ic = -3

# Plot: voltage vs. time
plt.figure(figsize=(8, 5))
plt.plot(t_sel_Ic, V_sel_Ic, marker=".", markersize=0.1, color="blue")
plt.axvline(t1_Ic, linestyle="--", linewidth=0.8, color="red", label="Transition points")
plt.axvline(t2_Ic, linestyle="--", linewidth=0.8, color="red")
plt.xlabel("Time t (ms)")
plt.ylabel("Voltage V (mV)")
plt.title("Voltage vs. time (integer flux)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PLOT_DIR / "plot5.png", format="png")
plt.show()

In [ ]:
# Current ramp from -I_pp/2 to +I_pp/2 between these times (ms)
t_ramp_start = -10
t_ramp_end = 10

# Reading uncertainty of the transition times (ms)
dt_Ic = 1.0

# Currents at the transition points (µA)
I1_Ic = -I_pp_int / 2 + I_pp_int / (t_ramp_end - t_ramp_start) * (t1_Ic - t_ramp_start)
I2_Ic = -I_pp_int / 2 + I_pp_int / (t_ramp_end - t_ramp_start) * (t2_Ic - t_ramp_start)
dI_Ic = I_pp_int / (t_ramp_end - t_ramp_start) * dt_Ic

# Critical current
I_c = (abs(I1_Ic) + abs(I2_Ic)) / 2  # µA
dI_c = np.sqrt(2) * dI_Ic / 2  # µA

print(f"I_1 = ({I1_Ic:.2f} ± {dI_Ic:.2f}) µA")
print(f"I_2 = ({I2_Ic:.2f} ± {dI_Ic:.2f}) µA")
print(f"I_c = ({I_c:.2f} ± {dI_c:.2f}) µA")

## Inverse mutual inductance of the feedback coil $M_{\Phi_B}^{-1}$

In [ ]:
# Load measurement data (time in ms, channel A: current ramp in V, channel B: voltage in mV)
data_B = np.loadtxt(
    DATA_DIR / "5_2_1" / "V_phi_peak_peak_max_center" / "V_phi_peak_peak_max_center_2.txt",
    skiprows=3,
)

# Time interval (ms)
t_min_B = -10
t_max_B = 10

# Calibrate current: the peak-to-peak ramp corresponds to I_pp = 180.06 µA
I_pp_B = 180.06  # µA
cal_B, rel_cal_B = calibrate_current(data_B[:, 1], I_pp_B)  # µA/V, relative uncertainty

# Extract measurement data
t_B = data_B[:, 0]  # ms
I_B = data_B[:, 1] * cal_B  # µA
V_B = data_B[:, 2]  # mV

# Select data within the time interval
mask_B = (t_B >= t_min_B) & (t_B <= t_max_B)
I_sel_B = I_B[mask_B]
V_sel_B = V_B[mask_B]

# Period boundaries (µA)
I1_B = -60
I2_B = 56

# Plot: V-I characteristic of the feedback coil
plt.figure(figsize=(8, 5))
plt.plot(I_sel_B, V_sel_B, marker=".", markersize=0.1, color="blue")
plt.axvline(I1_B, linestyle="--", linewidth=0.8, color="red", label="Period boundaries")
plt.axvline(I2_B, linestyle="--", linewidth=0.8, color="red")
plt.xlabel("Feedback coil current I (µA)")
plt.ylabel("Voltage V (mV)")
plt.title("Voltage-flux characteristic (feedback coil)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PLOT_DIR / "plot6.png", format="png")
plt.show()

In [ ]:
# Reading uncertainty of the period boundaries (µA)
dI_B = 2.0

# Number of flux quanta between the period boundaries
N_B = 3

# Inverse mutual inductance
M_inv_B = (I2_B - I1_B) / N_B  # µA/Φ0
dM_inv_read_B = np.sqrt(2) * dI_B / N_B  # µA/Φ0, reading of both boundaries
dM_inv_cal_B = M_inv_B * rel_cal_B  # µA/Φ0, current calibration
dM_inv_B = np.sqrt(dM_inv_read_B**2 + dM_inv_cal_B**2)  # µA/Φ0

print(f"Uncertainty components: reading {dM_inv_read_B:.3f} µA/Φ0, calibration {dM_inv_cal_B:.3f} µA/Φ0")

display(Math(
    rf"M_{{\Phi_B}}^{{-1}} = ({M_inv_B:.3f} \pm {dM_inv_B:.3f})"
    r"\,\mu\mathrm{A}/\Phi_0"
))

In [ ]:
# Convert current to flux; the offset is chosen, the absolute flux origin is not calibrated
phi_sel_B = I_sel_B / M_inv_B + 0.05  # Φ0

# Plot: V-Φ characteristic
plt.figure(figsize=(8, 5))
plt.plot(phi_sel_B, V_sel_B, marker=".", markersize=0.1, color="blue")
plt.xlabel(r"Flux $\Phi / \Phi_0$")
plt.ylabel("Voltage V (mV)")
plt.title("Voltage-flux characteristic (feedback coil)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PLOT_DIR / "plot7.png", format="png")
plt.show()

## Inverse mutual inductance of the input coil $M_{\mathrm{IN}}^{-1}$

In [ ]:
# Load measurement data (time in ms, channel A: current ramp in V, channel B: voltage in mV)
data_IN = np.loadtxt(
    DATA_DIR / "5_2_1" / "V_phi_x_peak_peak_max_center" / "V_phi_x_peak_peak_max_center_2.txt",
    skiprows=3,
)

# Time interval (ms)
t_min_IN = -10
t_max_IN = 10

# Calibrate current: the peak-to-peak ramp corresponds to I_pp = 100 µA
I_pp_IN = 100  # µA
cal_IN, rel_cal_IN = calibrate_current(data_IN[:, 1], I_pp_IN)  # µA/V, relative uncertainty

# Extract measurement data
t_IN = data_IN[:, 0]  # ms
I_IN = data_IN[:, 1] * cal_IN  # µA
V_IN = data_IN[:, 2]  # mV

# Select data within the time interval
mask_IN = (t_IN >= t_min_IN) & (t_IN <= t_max_IN)
I_sel_IN = I_IN[mask_IN]
V_sel_IN = V_IN[mask_IN]

# Period boundaries (µA)
I1_IN = -40.5
I2_IN = 39.5

# Plot: V-I characteristic of the input coil
plt.figure(figsize=(8, 5))
plt.plot(I_sel_IN, V_sel_IN, marker=".", markersize=0.1, color="blue")
plt.axvline(I1_IN, linestyle="--", linewidth=0.8, color="red", label="Period boundaries")
plt.axvline(I2_IN, linestyle="--", linewidth=0.8, color="red")
plt.xlabel("Input coil current I (µA)")
plt.ylabel("Voltage V (mV)")
plt.title("Voltage-flux characteristic (input coil)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PLOT_DIR / "plot8.png", format="png")
plt.show()

In [ ]:
# Reading uncertainty of the period boundaries (µA)
dI_IN = 2.0

# Number of flux quanta between the period boundaries
N_IN = 9

# Inverse mutual inductance
M_inv_IN = (I2_IN - I1_IN) / N_IN  # µA/Φ0
dM_inv_read_IN = np.sqrt(2) * dI_IN / N_IN  # µA/Φ0, reading of both boundaries
dM_inv_cal_IN = M_inv_IN * rel_cal_IN  # µA/Φ0, current calibration
dM_inv_IN = np.sqrt(dM_inv_read_IN**2 + dM_inv_cal_IN**2)  # µA/Φ0

print(f"Uncertainty components: reading {dM_inv_read_IN:.3f} µA/Φ0, calibration {dM_inv_cal_IN:.3f} µA/Φ0")

display(Math(
    rf"M_{{\mathrm{{IN}}}}^{{-1}} = ({M_inv_IN:.3f} \pm {dM_inv_IN:.3f})"
    r"\,\mu\mathrm{A}/\Phi_0"
))

In [ ]:
# Convert current to flux; the offset is chosen, the absolute flux origin is not calibrated
phi_sel_IN = I_sel_IN / M_inv_IN + 0.05  # Φ0

# Plot: V-Φ characteristic
plt.figure(figsize=(8, 5))
plt.plot(phi_sel_IN, V_sel_IN, marker=".", markersize=0.1, color="blue")
plt.xlabel(r"Flux $\Phi / \Phi_0$")
plt.ylabel("Voltage V (mV)")
plt.title("Voltage-flux characteristic (input coil)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PLOT_DIR / "plot9.png", format="png")
plt.show()

# 5.2.2 – Single-stage flux-locked loop (FLL)

## FLL characteristic

In [ ]:
# Time interval for the linear fit (ms)
t_min_FLL = -10
t_max_FLL = 10

# Peak-to-peak ramp current (µA)
I_pp_FLL = 20

fits_FLL = []
for n in (1, 2):
    # Load measurement data (time in ms, channel A: current ramp in V, channel B: output voltage in V)
    t_FLL, U_FLL, V_FLL = np.loadtxt(
        DATA_DIR / "5_2_2" / "single_FLL" / f"single_FLL_{n}.txt",
        skiprows=3,
    ).T

    # Calibrate current
    cal_FLL, rel_cal_FLL = calibrate_current(U_FLL, I_pp_FLL)  # µA/V, relative uncertainty
    I_FLL = U_FLL * cal_FLL  # µA

    # Linear fit within the time interval with input uncertainties
    mask_FLL = (t_FLL >= t_min_FLL) & (t_FLL <= t_max_FLL)
    sigma_I_FLL = resolution(U_FLL) / np.sqrt(12) * cal_FLL  # µA
    sigma_V_FLL = resolution(V_FLL) / np.sqrt(12)  # V
    m_FLL, b_FLL, dm_FLL, db_FLL, chi2_FLL = weighted_linear_fit(I_FLL[mask_FLL], V_FLL[mask_FLL], sigma_I_FLL, sigma_V_FLL)
    fits_FLL.append(dict(m=m_FLL, b=b_FLL, dm=dm_FLL, rel_cal=rel_cal_FLL))

    print(f"Measurement {n}: m = ({m_FLL:.4f} ± {dm_FLL:.4f}) V/µA, chi²_red = {chi2_FLL:.2f}, "
          f"calibration uncertainty {100 * rel_cal_FLL:.2f} %")

# Plot: FLL characteristic of the last measurement (_2)
plt.figure(figsize=(8, 5))
plt.plot(I_FLL, V_FLL, marker=".", linestyle="None", markersize=1, color="blue", label="Measured data")
plt.plot(I_FLL[mask_FLL], m_FLL * I_FLL[mask_FLL] + b_FLL, linewidth=2, color="red", label="Linear fit")
plt.xlabel("Input current I (µA)")
plt.ylabel("Output voltage V (V)")
plt.title("FLL characteristic")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PLOT_DIR / "plot10.png", format="png")
plt.show()

## FLL amplification

In [ ]:
# Feedback resistance (kΩ)
R_F_522 = 100

# Theoretical amplification: kΩ · µA/µA = mV/µA -> ×1e-3 for V/µA
A_theo = R_F_522 * (M_inv_B / M_inv_IN) * 1e-3  # V/µA
dA_theo = A_theo * np.sqrt((dM_inv_IN / M_inv_IN)**2 + (dM_inv_B / M_inv_B)**2)  # V/µA

# Experimental amplification: mean of both measurements
A_exp, dA_exp, u_stat, u_cal, u_rep = combine_repeats(
    [abs(fit["m"]) for fit in fits_FLL], [fit["dm"] for fit in fits_FLL], [fit["rel_cal"] for fit in fits_FLL]
)
rel_dA_exp = dA_exp / A_exp

# Deviation in standard deviations
n_sigma_A = np.abs(A_theo - A_exp) / np.sqrt(dA_exp**2 + dA_theo**2)

print(f"Theoretical amplification:  A_theo = ({A_theo:.3f} ± {dA_theo:.3f}) V/µA")
print(f"Experimental amplification: A_exp  = ({A_exp:.4f} ± {dA_exp:.4f}) V/µA")
print(f"  (stat. {u_stat:.5f}, calibration {u_cal:.4f}, reproducibility {u_rep:.4f}) V/µA")
print(f"Deviation:                  {n_sigma_A:.2f} σ")

# 5.2.3 – SQUID noise

## Noise spectrum

In [ ]:
# Load measurement data (frequency in MHz, channel B: voltage level in dBV)
data_noise = np.loadtxt(
    DATA_DIR / "5_2_3" / "30_kOhm_1_5_ghz" / "30_kOhm_1_5_ghz_2.txt",
    skiprows=3,
)

# Extract measurement data
f_noise = data_noise[:, 0]  # MHz
L_noise = data_noise[:, 1]  # dBV

# Plot: noise spectrum
plt.figure(figsize=(8, 5))
plt.plot(f_noise, L_noise, marker=".", linestyle="None", markersize=1, color="blue", label="Measured data")
plt.xscale("log")
plt.xlabel("Frequency f (MHz)")
plt.ylabel("Voltage level (dBV)")
plt.title("Noise spectrum")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PLOT_DIR / "plot11.png", format="png")
plt.show()

# 5.3.1 – Two-stage setup

## Voltage-flux characteristic ($\Phi_B$)

In [ ]:
# Load measurement data (time in ms, channel A: flux ramp in V, channel C: voltage in mV)
data_531 = np.loadtxt(
    DATA_DIR / "5_3_1" / "Phi_V_char_Phi_B" / "Phi_V_char_Phi_B_2.txt",
    skiprows=3,
)

# Extract measurement data
t_531 = data_531[:, 0]  # ms
U_ramp_531 = data_531[:, 1]  # V
V_531 = data_531[:, 2]  # mV

# Plot: voltage-flux characteristic
plt.figure(figsize=(8, 5))
plt.plot(U_ramp_531, V_531, linewidth=1.5, color="blue")
plt.xlabel("Flux ramp signal (V)")
plt.ylabel("Voltage V (mV)")
plt.title("Voltage-flux characteristic (two-stage setup)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PLOT_DIR / "plot12.png", format="png")
plt.show()

# 5.3.2 – Noise thermometry with the two-stage setup

## Transimpedance gain

In [ ]:
# The gain A_exp from 5.2.2 was measured with R_F = 100 kΩ.
# It scales linearly with the feedback resistance; this measurement used R_F = 30 kΩ.
R_F_532 = 30  # kΩ
gain_factor = R_F_532 / R_F_522

A_532 = gain_factor * A_exp  # V/µA
dA_532 = gain_factor * dA_exp  # V/µA

print(f"Transimpedance gain: A = ({A_532:.4f} ± {dA_532:.4f}) V/µA")

## Current noise spectrum and fit

In [ ]:
# Load measurement data (frequency in MHz, channel C: voltage level in dBV)
# The files _1 and _2 of this measurement are identical, so only _1 is evaluated
data_532 = np.loadtxt(
    DATA_DIR / "5_3_2" / "two_stage_30kO_4GHz" / "two_stage_30kO_4GHz_1.txt",
    skiprows=3,
)

# Extract measurement data
f_532 = data_532[:, 0] * 1e6  # Hz
L_532 = data_532[:, 1]  # dBV

# Voltage noise density: sqrt(S_V) = 10^(L/20) / sqrt(Δf)
delta_f_532 = np.abs(f_532[2] - f_532[1])  # Hz, frequency resolution
sqrtS_V_532 = 10 ** (L_532 / 20) / np.sqrt(delta_f_532)  # V/√Hz

# Current noise density: V/µA -> V/A with ×1e6
sqrtS_I_532 = sqrtS_V_532 / (A_532 * 1e6)  # A/√Hz

# Input inductance (H)
L_IN = 2.9e-9

# Model: thermal current noise of a resistor R, low-pass filtered by L_IN
def sqrt_S_I(f, T, R):
    f_c = R / (2 * np.pi * L_IN)  # Hz
    return np.sqrt(4 * K_BOLTZMANN * T / R / (1 + (f / f_c)**2))  # A/√Hz

# Fit range (indices)
i_min_532 = 50
i_max_532 = 3500

# The fit runs in rescaled units (√S_I in pA/√Hz, R in mΩ), because the raw values are
# very small and the fit is numerically ill-conditioned otherwise
def sqrt_S_I_scaled(f, T, R_mOhm):
    return sqrt_S_I(f, T, R_mOhm * 1e-3) * 1e12  # pA/√Hz

# Input uncertainty per frequency bin: relative scatter of neighbouring bins in the fit range
y_fit_532 = sqrtS_I_532[i_min_532:i_max_532]
rel_diff_532 = np.diff(y_fit_532) / (0.5 * (y_fit_532[1:] + y_fit_532[:-1]))
sigma_rel_532 = np.std(rel_diff_532) / np.sqrt(2)

def fit_noise(i_min, i_max):
    """Weighted fit of the current noise spectrum; σ_i = σ_rel · model (two iterations)."""
    f_fit = f_532[i_min:i_max]
    y_fit = sqrtS_I_532[i_min:i_max] * 1e12  # pA/√Hz
    popt, _ = curve_fit(sqrt_S_I_scaled, f_fit, y_fit, p0=[1.0, 10.0], bounds=([0, 0], [np.inf, np.inf]))
    sigma = sigma_rel_532 * sqrt_S_I_scaled(f_fit, *popt)
    popt, pcov = curve_fit(
        sqrt_S_I_scaled, f_fit, y_fit, p0=popt, sigma=sigma, absolute_sigma=True,
        bounds=([0, 0], [np.inf, np.inf]),  # T > 0, R > 0
    )
    chi2_red = np.sum(((y_fit - sqrt_S_I_scaled(f_fit, *popt)) / sigma)**2) / (len(f_fit) - 2)
    perr = np.sqrt(np.diag(pcov)) * np.sqrt(max(1.0, chi2_red))
    return popt, perr, chi2_red

(T_532, R_532), (dT_stat_532, dR_stat_532), chi2_532 = fit_noise(i_min_532, i_max_532)  # K, mΩ

# Systematic uncertainty from the choice of the fit range
fit_ranges_532 = [(30, 3500), (100, 3500), (50, 3000), (50, 4000)]
T_var_532, R_var_532 = np.array([fit_noise(i_min, i_max)[0] for i_min, i_max in fit_ranges_532]).T
dT_range_532 = np.max(np.abs(T_var_532 - T_532))  # K
dR_range_532 = np.max(np.abs(R_var_532 - R_532))  # mΩ

# Systematic uncertainty from the transimpedance gain: S_I ∝ 1/A² -> T ∝ 1/A², R independent of A
dT_gain_532 = T_532 * 2 * dA_532 / A_532  # K

# Systematic uncertainty from the input inductance: R = 2π L f_c ∝ L and T ∝ R ∝ L
dL_IN = 0.0  # H, enter the uncertainty of L_IN here if it is known
dT_L_532 = T_532 * dL_IN / L_IN  # K
dR_L_532 = R_532 * dL_IN / L_IN  # mΩ

# Total uncertainties
dT_532 = np.sqrt(dT_stat_532**2 + dT_range_532**2 + dT_gain_532**2 + dT_L_532**2)  # K
dR_532 = np.sqrt(dR_stat_532**2 + dR_range_532**2 + dR_L_532**2)  # mΩ
f_c_532 = R_532 * 1e-3 / (2 * np.pi * L_IN)  # Hz
df_c_532 = f_c_532 * np.sqrt(dR_stat_532**2 + dR_range_532**2) / R_532  # Hz, independent of L_IN

print(f"Relative scatter per bin: {100 * sigma_rel_532:.1f} %, chi²_red = {chi2_532:.2f}")
print(f"Temperature:      T   = ({T_532:.2f} ± {dT_532:.2f}) K")
print(f"  (stat. {dT_stat_532:.3f} K, fit range {dT_range_532:.3f} K, gain {dT_gain_532:.3f} K, L_IN {dT_L_532:.3f} K)")
print(f"Resistance:       R   = ({R_532:.2f} ± {dR_532:.2f}) mΩ")
print(f"  (stat. {dR_stat_532:.3f} mΩ, fit range {dR_range_532:.3f} mΩ, L_IN {dR_L_532:.3f} mΩ)")
print(f"Cutoff frequency: f_c = ({f_c_532 / 1e3:.0f} ± {df_c_532 / 1e3:.0f}) kHz")

f_fit_532 = f_532[i_min_532:i_max_532]

# Plot: current noise spectrum with fit
plt.figure(figsize=(12, 7))
plt.loglog(f_532 / 1e6, sqrtS_I_532, color="dodgerblue", label="Measured data")
plt.loglog(f_fit_532 / 1e6, sqrt_S_I(f_fit_532, T_532, R_532 * 1e-3), linewidth=3, color="tomato", label="Fit")
plt.axvline(f_532[i_min_532] / 1e6, linestyle="--", color="gray", label="Fit interval")
plt.axvline(f_532[i_max_532 - 1] / 1e6, linestyle="--", color="gray")
plt.text(
    0.02, 0.05,
    rf"$T = ({T_532:.2f} \pm {dT_532:.2f})$ K" "\n" rf"$R = ({R_532:.2f} \pm {dR_532:.2f})$ m$\Omega$",
    transform=plt.gca().transAxes, fontsize=12,
)
plt.xlim(1e-3, 5)
plt.xlabel("Frequency f (MHz)", size=14)
plt.ylabel(r"Current noise density $\sqrt{S_I}$ (A/$\sqrt{\mathrm{Hz}}$)", size=14)
plt.title(r"Current noise spectrum (two-stage setup, $R_F$ = 30 k$\Omega$)", size=16)
plt.legend(loc="best")
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.savefig(PLOT_DIR / "plot13.png", format="png")
plt.show()